# Sentiment Analysis Using BiLSTM

This project performs sentiment analysis on the IMDb 50K Movie Reviews dataset using a Bidirectional Long Short-Term Memory (BiLSTM) neural network.

Objective:
To classify movie reviews as Positive or Negative based on their text content.

In [1]:
import os
print(os.path.getsize('IMDB Dataset.csv'))

66212309


# Dataset Loading

The IMDb 50K Movie Reviews dataset is loaded using Pandas.

The dataset contains:
- 50,000 movie reviews
- 25,000 positive reviews
- 25,000 negative reviews

The dataset is balanced and suitable for sentiment classification.

In [2]:
import pandas as pd

df = pd.read_csv('IMDB Dataset.csv')

print(df.shape)

(50000, 2)


# Data Exploration

The first few records are displayed to understand the structure of the dataset.

The sentiment distribution is checked to verify whether the dataset is balanced.

In [3]:
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [4]:
print(df['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


# Label Encoding

Machine learning models cannot process text labels directly.

Therefore:
Positive → 1
Negative → 0

This process is known as Label Encoding.

In [6]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

print(df.head())

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1


# Data Cleaning

Movie reviews may contain HTML tags such as <br />.

These tags do not contribute to sentiment and are removed to improve data quality.

In [8]:
import re

def remove_html(text):
    return re.sub(r'<.*?>', '', text)

df['review'] = df['review'].apply(remove_html)

# Text Normalization

All review text is converted to lowercase.

This ensures that words such as:
Good, GOOD, good

are treated as the same word.

In [9]:
df['review'] = df['review'].str.lower()

print(df['review'][1])

a wonderful little production. the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. the actors are extremely well chosen- michael sheen not only "has got all the polari" but he has all the voices down pat too! you can truly see the seamless editing guided by the references to williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. a masterful production about one of the great master's of comedy and his life. the realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. it plays on our knowledge and our senses, particularly with the scenes concerning orton and halliwell and the sets (particularly of their flat with halliwell's murals decorating every surface) are terribly well done.


# Tokenization

Neural networks cannot understand text directly.

Tokenization converts words into numerical values so that the model can process them.

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(df['review'])

sequences = tokenizer.texts_to_sequences(df['review'])

print(sequences[0][:20])

[26, 4, 1, 78, 2099, 44, 1071, 11, 99, 146, 38, 309, 3182, 397, 473, 25, 3190, 32, 22, 201]


# Padding

Movie reviews have different lengths.

Padding ensures that all reviews have the same length (200 tokens) before being fed into the neural network.

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = pad_sequences(sequences, maxlen=200)

print(X.shape)

(50000, 200)


# Train-Test Split

The dataset is divided into:

- 80% Training Data
- 20% Testing Data

Training data is used to learn patterns.
Testing data is used to evaluate model performance.

In [12]:
from sklearn.model_selection import train_test_split

y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(40000, 200)
(10000, 200)


# BiLSTM Model Architecture

The model consists of:

1. Embedding Layer
   - Converts word IDs into dense vectors.

2. Bidirectional LSTM Layer
   - Processes text from both directions.
   - Captures contextual information effectively.

3. Dense Layer
   - Produces the final sentiment prediction.

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense

model = Sequential([
    Embedding(input_dim=10000, output_dim=128),
    Bidirectional(LSTM(64)),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Model Training

The BiLSTM model is trained for 5 epochs.

During training:
- Accuracy measures training performance.
- Validation Accuracy measures performance on unseen data.

In [14]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 303s 475ms/step - accuracy: 0.8344 - loss: 0.3654 - val_accuracy: 0.8800 - val_loss: 0.2870
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 295s 472ms/step - accuracy: 0.9086 - loss: 0.2333 - val_accuracy: 0.8985 - val_loss: 0.2472
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 314s 503ms/step - accuracy: 0.9332 - loss: 0.1749 - val_accuracy: 0.8903 - val_loss: 0.2773
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 312s 499ms/step - accuracy: 0.9492 - loss: 0.1384 - val_accuracy: 0.8871 - val_loss: 0.2858
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 289s 463ms/step - accuracy: 0.9610 - loss: 0.1082 - val_accuracy: 0.8893 - val_loss: 0.3490


# Model Evaluation

The trained model is evaluated on the test dataset.

Performance is measured using classification accuracy.

In [15]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 64ms/step - accuracy: 0.8893 - loss: 0.3490
Test Accuracy: 0.8892999887466431


## Conclusion

A Sentiment Analysis system was developed using a Bidirectional LSTM (BiLSTM) model on the IMDb 50K Movie Reviews dataset.

After preprocessing, tokenization, padding, and training, the model achieved a test accuracy of 88.93%.

The results demonstrate that BiLSTM is effective for sentiment classification tasks.

In [16]:
model.save("sentiment_bilstm_model.keras")